# 01 — Исследование данных (EDA)

Цель: понять распределение классов, качество изображений, статистику датасета DENTEX.

In [2]:

import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

META     = Path('../data/processed/metadata.csv')
IMG_DIR  = Path('../data/processed/images')
MASK_DIR = Path('../data/processed/masks')

print('cv2 version:', cv2.__version__)
print('IMG_DIR exists:', IMG_DIR.exists(), '| images:', len(list(IMG_DIR.glob('*.png'))))
print('MASK_DIR exists:', MASK_DIR.exists(), '| masks:', len(list(MASK_DIR.glob('*.png'))))

cv2 version: 4.13.0
IMG_DIR exists: True | images: 0
MASK_DIR exists: True | masks: 0


In [3]:
if not META.exists():
    print('metadata.csv not found — run: python scripts/preprocess.py')
else:
    df = pd.read_csv(META)
    print(f'Total images: {len(df)}')
    display(df.head())

metadata.csv not found — run: python scripts/preprocess.py


In [4]:
# Class distribution in masks
mask_files = list(MASK_DIR.glob('*.png'))[:200]
if not mask_files:
    print('No masks found — run: python scripts/preprocess.py')
else:
    class_names = ['background', 'tooth', 'bone', 'cortical']
    pixel_counts = np.zeros(4, dtype=np.int64)

    for mask_path in mask_files:
        m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if m is None:
            print(f'Warning: cannot read {mask_path.name}, skipping')
            continue
        for c in range(4):
            pixel_counts[c] += (m == c).sum()

    total = pixel_counts.sum()
    if total == 0:
        print('All masks are empty or unreadable')
    else:
        fracs = pixel_counts / total
        plt.figure(figsize=(8, 4))
        plt.bar(class_names, fracs * 100, color=['#888', '#e55', '#55e', '#ee5'])
        plt.ylabel('% пикселей')
        plt.title('Распределение классов в масках')
        plt.tight_layout()
        plt.show()

        print('Веса классов (обратная частота):')
        inv = 1.0 / (fracs + 1e-6)
        inv_norm = inv / inv.sum()
        for name, f, w in zip(class_names, fracs, inv_norm):
            print(f'  {name:12s}: {f:.4%} пикселей  →  вес {w:.4f}')

No masks found — run: python scripts/preprocess.py


In [5]:
# Примеры изображений + маски
samples = list(IMG_DIR.glob('*.png'))[:6]
if not samples:
    print('No images found in', IMG_DIR)
else:
    n = len(samples)
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    if n == 1:
        axes = axes.reshape(2, 1)  # гарантируем 2D при n=1

    for i, img_path in enumerate(samples):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        mask_path = MASK_DIR / img_path.name
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else None

        if img is None:
            print(f'Cannot read {img_path.name}')
            continue

        axes[0, i].imshow(img, cmap='gray')
        axes[0, i].set_title(img_path.stem[:10], fontsize=8)
        axes[0, i].axis('off')

        if mask is not None:
            axes[1, i].imshow(mask, cmap='tab10', vmin=0, vmax=3)
        else:
            axes[1, i].text(0.5, 0.5, 'no mask', ha='center', va='center')
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Image')
    axes[1, 0].set_ylabel('Mask')
    plt.suptitle('ОПТГ-примеры + маски сегментации')
    plt.tight_layout()
    plt.show()

No images found in ..\data\processed\images


In [ ]:
# Статистика яркости изображений
img_files = list(IMG_DIR.glob('*.png'))[:100]
if not img_files:
    print('No images found — run: python scripts/preprocess.py')
else:
    means, stds = [], []
    for img_path in img_files:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        arr = img.astype(float) / 255.0
        means.append(arr.mean())
        stds.append(arr.std())

    print(f'Средняя яркость: {np.mean(means):.3f} ± {np.std(means):.3f}')
    print(f'Средний контраст: {np.mean(stds):.3f} ± {np.std(stds):.3f}')
    print(f'Обработано снимков: {len(means)} / {len(img_files)}')